# CSP-Atlas — Module 1 Version 1
## Programmatic Prompt Generation & Comprehension Filtering

**Project:** Topology and Modularity of Sparse Transformers  
**Source of truth:** Project Atlas 2, Sections 2 and 9  
**GitHub:** https://github.com/piotrwilam/CSP-Atlas  
**Checkpoints:** `/content/drive/MyDrive/DATA/CSP-Atlas`  
**HuggingFace:** https://huggingface.co/CSP-Atlas  

---

**Three modes:**
- `MODE = "test"` → 5×5, 10 kept (250 prompts, ~5-10 min)
- `MODE = "small"` → ~40×50, 50 kept (~100K prompts, ~4-8 hrs)
- `MODE = "full"` → 125×153, 100 kept (~1.9M prompts, ~20-40 hrs)

In [ ]:
# version 1.11
# Cell 1 — Install dependencies (run once per Colab session / per Python environment)
import subprocess, sys, os, types, importlib.util, glob, inspect

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'pyarrow', 'huggingface_hub'], check=True)

# Use the transformers_modules cached gpt.py if available, else hub download
tm_base     = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/openai")
tm_gpt_hits = glob.glob(os.path.join(tm_base, "*/gpt.py"))

if tm_gpt_hits:
    gpt_path  = tm_gpt_hits[0]
    hook_path = os.path.join(os.path.dirname(gpt_path), "hook_utils.py")
    print(f"Using transformers_modules cache: {os.path.dirname(gpt_path)}")
else:
    from huggingface_hub import hf_hub_download
    gpt_path  = hf_hub_download("openai/circuit-sparsity", "gpt.py")
    hook_path = hf_hub_download("openai/circuit-sparsity", "hook_utils.py")
    print("Using hf_hub_download (first run)")

hf_dir = os.path.dirname(gpt_path)

# Register circuit_sparsity package
cs_pkg = types.ModuleType('circuit_sparsity')
cs_pkg.__path__ = [hf_dir]
cs_pkg.__package__ = 'circuit_sparsity'
sys.modules['circuit_sparsity'] = cs_pkg

# Load hook_utils as circuit_sparsity.hook_utils
hook_spec = importlib.util.spec_from_file_location('circuit_sparsity.hook_utils', hook_path)
hook_mod  = importlib.util.module_from_spec(hook_spec)
hook_mod.__package__ = 'circuit_sparsity'
sys.modules['circuit_sparsity.hook_utils'] = hook_mod
hook_spec.loader.exec_module(hook_mod)
cs_pkg.hook_utils = hook_mod

# Load gpt.py to get the real GPTConfig (with all proper defaults like Linear, cat_pos_emb, etc.)
gpt_spec = importlib.util.spec_from_file_location('circuit_sparsity.gpt', gpt_path)
gpt_mod  = importlib.util.module_from_spec(gpt_spec)
gpt_mod.__package__ = 'circuit_sparsity'
sys.modules['circuit_sparsity.gpt'] = gpt_mod
gpt_spec.loader.exec_module(gpt_mod)

# Patch GPTConfig to silently ignore unknown kwargs (e.g. unembed_rank from config.py).
# Subclassing preserves all real defaults (Linear, cat_pos_emb, dropout, etc.)
_RealGPTConfig = gpt_mod.GPTConfig
_valid_params  = set(inspect.signature(_RealGPTConfig.__init__).parameters) - {'self'}

class _PatchedGPTConfig(_RealGPTConfig):
    def __init__(self, **kwargs):
        super().__init__(**{k: v for k, v in kwargs.items() if k in _valid_params})

gpt_mod.GPTConfig = _PatchedGPTConfig
cs_pkg.gpt = gpt_mod

print(f"circuit_sparsity.gpt ready — GPTConfig patched (filters unknown kwargs)")

In [ ]:
# Cell 2 — Configuration (edit MODE here before running)
# ============================================================
MODE = "test"  # "test", "small", or "full"

# Paths
MODEL_NAME = "openai/circuit-sparsity"
HF_REPO = "CSP-Atlas"

# ── Detect environment ───────────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DATA_DIR = "/content/drive/MyDrive/DATA/CSP-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/CSP-Atlas"

CHECKPOINT_DIR = DATA_DIR

# --- Per-mode defaults ---
if MODE == "test":
    N_GENERATE = 10
    N_KEEP = 10
    CHECKPOINT_EVERY = 100
    CATASTROPHIC_THRESHOLD = 3.0

elif MODE == "small":
    N_GENERATE = 50
    N_KEEP = 50
    CHECKPOINT_EVERY = 100
    CATASTROPHIC_THRESHOLD = 3.0

elif MODE == "full":
    N_GENERATE = 100
    N_KEEP = 100
    CHECKPOINT_EVERY = 50
    CATASTROPHIC_THRESHOLD = 3.0

print(f"Mode: {MODE}")
print(f"Data dir: {DATA_DIR}")
print(f"Generate {N_GENERATE} → Keep {N_KEEP} per (AST, builtin) pair")

In [ ]:
# Cell 3 — Imports
import ast
import builtins as builtins_module
import random
import json
import logging
import os
import sys
import textwrap
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("module1")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 4 — Mount Google Drive (Colab only)
if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

## Variance Schema & Concept Matrix

**Atlas 2 §9:** *"Your 100 variations must be maximally orthogonal. Inject three types of variance: Lexical/Semantic, Structural/Contextual, and Padding."*

In [ ]:
# Cell 6 — Variance Schema (5 semantic domains)
DOMAINS = {
    "finance": {
        "var_names": {
            "list": "ledger_entries", "dict": "account_records",
            "item": "transaction", "func": "audit_record",
            "class": "PortfolioManager", "method": "calculate_returns",
            "value": "balance", "key": "ticker",
        },
        "mock_data": {
            "list": "[1050.50, -20.00, 400.25, 88.10]",
            "dict": "{'ticker': 'AAPL', 'shares': 150, 'price': 178.50}",
            "int": "42500", "str": "'USD-2024-Q3-REPORT'",
            "float": "3.75", "set": "{'NYSE', 'NASDAQ', 'LSE'}",
            "tuple": "('BUY', 150, 178.50)", "bool": "True",
        },
    },
    "biology": {
        "var_names": {
            "list": "dna_samples", "dict": "genome_annotations",
            "item": "genome_sequence", "func": "analyze_genome",
            "class": "SequenceAnalyzer", "method": "run_alignment",
            "value": "mutation_rate", "key": "gene_id",
        },
        "mock_data": {
            "list": "['ACTG', 'GCTA', 'CGAT', 'TTAC']",
            "dict": "{'gene': 'BRCA1', 'chromosome': 17, 'position': 43044295}",
            "int": "43044295", "str": "'BRCA1-VARIANT-P53'",
            "float": "0.95", "set": "{'adenine', 'thymine', 'guanine'}",
            "tuple": "('chr17', 43044295, 43125483)", "bool": "False",
        },
    },
    "gaming": {
        "var_names": {
            "list": "player_scores", "dict": "character_stats",
            "item": "player_entry", "func": "update_leaderboard",
            "class": "GameEngine", "method": "spawn_entity",
            "value": "hit_points", "key": "player_id",
        },
        "mock_data": {
            "list": "[9500, 8700, 12400, 6300]",
            "dict": "{'health': 100, 'mana': 50, 'attack': 25}",
            "int": "9500", "str": "'LEVEL_42_BOSS'",
            "float": "1.5", "set": "{'warrior', 'mage', 'rogue'}",
            "tuple": "('dragon', 5000, 'fire')", "bool": "True",
        },
    },
    "physics": {
        "var_names": {
            "list": "measurements", "dict": "particle_data",
            "item": "reading", "func": "compute_trajectory",
            "class": "SimulationRunner", "method": "step_forward",
            "value": "velocity", "key": "particle_id",
        },
        "mock_data": {
            "list": "[9.81, 3.0e8, 6.674e-11, 1.602e-19]",
            "dict": "{'mass': 1.67e-27, 'charge': 1.6e-19, 'spin': 0.5}",
            "int": "299792458", "str": "'MUON_DECAY_EVENT'",
            "float": "9.81", "set": "{'proton', 'neutron', 'electron'}",
            "tuple": "(0.0, 9.81, -3.2)", "bool": "False",
        },
    },
    "ecommerce": {
        "var_names": {
            "list": "shopping_cart", "dict": "product_catalog",
            "item": "order_item", "func": "process_checkout",
            "class": "OrderProcessor", "method": "apply_discount",
            "value": "total_price", "key": "sku",
        },
        "mock_data": {
            "list": "[29.99, 15.50, 89.00, 4.99]",
            "dict": "{'sku': 'WDG-4420', 'price': 29.99, 'stock': 142}",
            "int": "142", "str": "'ORDER-2024-XK9001'",
            "float": "29.99", "set": "{'shipped', 'pending', 'delivered'}",
            "tuple": "('WDG-4420', 29.99, 3)", "bool": "True",
        },
    },
}

DOMAIN_KEYS = list(DOMAINS.keys())
WRAPPER_TYPES = ["global", "function", "method"]
WRAPPER_WEIGHTS = [0.40, 0.30, 0.30]

PADDING_BEFORE = [
    "", "result = None", "print('Starting process')",
    "status = True", "counter = 0",
]
PADDING_AFTER = [
    "", "print('Done')", "status = False",
    "counter += 1", "result = None",
]

print(f"Domains: {DOMAIN_KEYS}")

In [ ]:
# Cell 7 — Concept Matrix / Stage A  (sparse compatibility map)

# ---------------------------------------------------------------------------
# AST node families
# ---------------------------------------------------------------------------
AST_FAMILIES = {
    "iteration":   ["For", "While", "AsyncFor", "ListComp", "SetComp",
                    "DictComp", "GeneratorExp"],
    "exception":   ["Try", "Raise", "ExceptHandler"],
    "callable":    ["FunctionDef", "AsyncFunctionDef", "Return",
                    "Call", "Lambda", "Yield", "YieldFrom"],
    "class":       ["ClassDef"],
    "assignment":  ["Assign", "AugAssign", "AnnAssign", "Delete",
                    "Global", "Nonlocal"],
    "conditional": ["If", "IfExp"],
    "operator":    ["BinOp", "BoolOp", "UnaryOp", "Compare"],
    "structural":  ["With", "AsyncWith"],
    "import":      ["Import", "ImportFrom"],
    "access":      ["Attribute", "Subscript", "Starred", "Slice"],
    "literal":     ["Dict", "Set"],
    "isolated":    ["Pass", "Break", "Continue", "Assert"],
}

# ---------------------------------------------------------------------------
# Builtin families
# ---------------------------------------------------------------------------
BUILTIN_FAMILIES = {
    "iterables":   ["list", "tuple", "set", "frozenset", "dict",
                    "str", "bytes", "bytearray", "range", "memoryview"],
    "numerics":    ["int", "float", "complex", "bool"],
    "callables":   ["print", "len", "range", "enumerate", "zip", "map",
                    "filter", "sorted", "reversed", "min", "max", "sum",
                    "abs", "round", "any", "all", "isinstance", "issubclass",
                    "hasattr", "getattr", "setattr", "delattr", "callable",
                    "iter", "next", "hash", "id", "repr", "input", "open",
                    "super", "property", "staticmethod", "classmethod",
                    "vars", "dir", "type", "object"],
    "exceptions":  ["Exception", "ValueError", "TypeError", "KeyError",
                    "IndexError", "AttributeError", "RuntimeError",
                    "StopIteration", "FileNotFoundError", "OSError",
                    "ImportError", "NotImplementedError", "ZeroDivisionError",
                    "NameError", "OverflowError", "MemoryError",
                    "RecursionError", "ArithmeticError", "LookupError"],
    "types":       ["int", "float", "str", "bool", "list", "dict",
                    "tuple", "set", "bytes", "type", "object", "complex"],
}

# ---------------------------------------------------------------------------
# Compatibility map: which builtin families each AST family may pair with
# ---------------------------------------------------------------------------
COMPATIBILITY = {
    "iteration":   ["iterables"],
    "exception":   ["exceptions"],
    "callable":    ["callables", "types", "iterables", "numerics"],
    "class":       ["callables", "types", "iterables", "numerics", "exceptions"],
    "assignment":  ["iterables", "numerics", "types", "callables"],
    "conditional": ["iterables", "numerics", "types", "exceptions", "callables"],
    "operator":    ["numerics", "iterables"],
    "structural":  ["callables", "types"],
    "import":      [],          # Import/ImportFrom need no builtin argument
    "access":      ["iterables", "types", "callables"],
    "literal":     ["iterables", "numerics"],
    "isolated":    [],          # Pass/Break/Continue tested in isolation
}

# ---------------------------------------------------------------------------
# Build the sparse concept matrix
# ---------------------------------------------------------------------------
def get_ast_nodes() -> list[str]:
    abstract = {
        'AST', 'mod', 'stmt', 'expr', 'expr_context', 'boolop',
        'operator', 'unaryop', 'cmpop', 'comprehension', 'excepthandler',
        'arguments', 'arg', 'keyword', 'alias', 'withitem',
        'match_case', 'pattern', 'type_ignore', 'type_param',
        'ParamSpec', 'TypeVar', 'TypeVarTuple',
    }
    nodes = []
    for name in dir(ast):
        obj = getattr(ast, name)
        if (isinstance(obj, type) and issubclass(obj, ast.AST)
                and name not in abstract and not name.startswith('_')):
            nodes.append(name)
    return sorted(nodes)

def get_builtin_objects() -> list[str]:
    return sorted([n for n in dir(builtins_module) if not n.startswith('_')])

def build_sparse_matrix(mode: str) -> list[tuple[str, str]]:
    all_ast      = set(get_ast_nodes())
    all_builtins = set(get_builtin_objects())

    # Reverse lookup: builtin name → family set
    builtin_to_families = {}
    for fam, members in BUILTIN_FAMILIES.items():
        for m in members:
            builtin_to_families.setdefault(m, set()).add(fam)

    # Reverse lookup: ast name → family
    ast_to_family = {}
    for fam, members in AST_FAMILIES.items():
        for m in members:
            ast_to_family[m] = fam

    pairs = []
    for ast_fam, builtin_fams in COMPATIBILITY.items():
        ast_nodes = [n for n in AST_FAMILIES[ast_fam] if n in all_ast]

        if not builtin_fams:
            # isolated nodes — one sentinel pair so the generator runs them
            for node in ast_nodes:
                pairs.append((node, "_isolated_"))
            continue

        allowed_builtins = set()
        for bf in builtin_fams:
            allowed_builtins |= set(BUILTIN_FAMILIES[bf])
        allowed_builtins &= all_builtins  # only keep what actually exists

        for node in ast_nodes:
            for b in sorted(allowed_builtins):
                pairs.append((node, b))

    # Apply mode subsets
    if mode == "test":
        test_ast      = {"For", "If", "ListComp", "Try", "FunctionDef"}
        test_builtins = {"list", "dict", "int", "ValueError", "range"}
        pairs = [(a, b) for a, b in pairs
                 if a in test_ast and (b in test_builtins or b == "_isolated_")]
    elif mode == "small":
        small_ast = set([
            "FunctionDef", "AsyncFunctionDef", "ClassDef", "Return", "Delete",
            "Assign", "AugAssign", "AnnAssign", "For", "AsyncFor", "While", "If",
            "With", "AsyncWith", "Raise", "Try", "Assert", "Import", "ImportFrom",
            "Global", "Nonlocal", "Pass", "Break", "Continue",
            "BoolOp", "BinOp", "UnaryOp", "Lambda", "IfExp", "Dict", "Set",
            "ListComp", "SetComp", "DictComp", "GeneratorExp", "Yield", "YieldFrom",
            "Compare", "Call", "Attribute", "Subscript", "Starred", "Slice",
        ])
        small_builtins = set([
            "int", "float", "str", "bool", "list", "dict", "tuple", "set",
            "frozenset", "bytes", "bytearray", "complex", "object", "type",
            "memoryview", "print", "len", "range", "enumerate", "zip", "map",
            "filter", "sorted", "reversed", "min", "max", "sum", "abs", "round",
            "any", "all", "isinstance", "issubclass", "hasattr", "getattr",
            "setattr", "delattr", "callable", "iter", "next", "hash", "id",
            "repr", "input", "open", "super", "property", "staticmethod",
            "classmethod", "Exception", "ValueError", "TypeError", "KeyError",
            "IndexError", "AttributeError", "RuntimeError", "StopIteration",
            "FileNotFoundError", "OSError", "ImportError", "NotImplementedError",
            "ZeroDivisionError",
        ])
        pairs = [(a, b) for a, b in pairs
                 if a in small_ast and (b in small_builtins or b == "_isolated_")]

    return pairs

concept_pairs = build_sparse_matrix(MODE)

print(f"Mode           : {MODE}")
print(f"Total pairs    : {len(concept_pairs)}")
print(f"  (vs naive    : {len(get_ast_nodes()) * len(get_builtin_objects())} full Cartesian)")
print(f"\nSample pairs:")
for p in concept_pairs[:10]:
    print(f"  {p}")

## AST Prompt Generator (Stage B)

**Atlas 2 §2:** *"Use `ast.parse()` to build templates and `ast.unparse()` to yield the final string. Do not rely on raw f-string concatenation for the final output."*

In [ ]:
# Cell 8 — AST Prompt Generator / Stage B (+ smoke test)
class ASTPromptGenerator:

    def __init__(self, domains: dict):
        self.domains = domains
        self.domain_keys = list(domains.keys())

    def generate_batch(self, ast_node: str, builtin_obj: str, n: int = 150) -> list[dict]:
        results = []
        for i in range(n):
            try:
                dk = self.domain_keys[i % len(self.domain_keys)]
                domain = self.domains[dk]
                d, m = domain["var_names"], domain["mock_data"]

                essence = self._build_essence(ast_node, builtin_obj, d, m)
                if essence is None:
                    continue

                pad_b = random.choice(PADDING_BEFORE)
                pad_a = random.choice(PADDING_AFTER)
                body = "\n".join(p for p in [pad_b, essence, pad_a] if p)

                wrapper = random.choices(WRAPPER_TYPES, weights=WRAPPER_WEIGHTS, k=1)[0]
                wrapped = self._apply_wrapper(body, wrapper, d)

                tree = ast.parse(wrapped)
                prompt_text = ast.unparse(tree)
                verified = self._verify(tree, ast_node)

                results.append({
                    "prompt_text": prompt_text,
                    "ast_verified": verified,
                    "domain": dk,
                    "wrapper": wrapper,
                })
            except SyntaxError:
                continue
            except Exception as e:
                logger.debug(f"Error ({ast_node}, {builtin_obj}) #{i}: {e}")
                continue

        if len(results) == 0:
            logger.warning(f"ZERO prompts for ({ast_node}, {builtin_obj})")
        return results

    def _build_essence(self, node, bobj, d, m):
        # Isolated nodes have no builtin argument — use a generic wrapper
        if bobj == "_isolated_":
            return self._isolated_tpl(node, d)

        def mock(t=None):
            return m.get(t or bobj, m.get("list", "[1, 2, 3]"))

        def iterable_expr():
            if bobj in ("list", "tuple", "set", "frozenset"):
                return f"{d['list']} = {mock('list')}"
            elif bobj == "dict":
                return f"{d['dict']} = {mock('dict')}"
            elif bobj == "range":
                return f"{d['list']} = list(range(10))"
            elif bobj == "str":
                return f"{d['list']} = list({mock('str')})"
            elif bobj == "int":
                return f"{d['list']} = list(range({mock('int')} % 20))"
            elif bobj in ("bytes", "bytearray"):
                return f"{d['list']} = list(b'hello')"
            else:
                return f"{d['list']} = [str({bobj}), str(type({bobj}))]"

        T = {
            "For": lambda: f"""{iterable_expr()}
for {d['item']} in {d['list']}:
    {d['func']}({d['item']})""",

            "While": lambda: f"""{d['value']} = {mock('int')}
while {d['value']} > 0:
    {d['func']}({d['value']})
    {d['value']} -= 1""",

            "AsyncFor": lambda: f"""async def _run():
    {iterable_expr()}
    async for {d['item']} in {d['list']}:
        {d['func']}({d['item']})""",

            "If": lambda: f"""{d['value']} = {mock()}
if isinstance({d['value']}, {bobj}):
    {d['func']}({d['value']})
else:
    pass""",

            "IfExp": lambda: f"""{d['value']} = {mock()}
result = {d['func']}({d['value']}) if isinstance({d['value']}, {bobj}) else None""",

            "ListComp": lambda: f"""{iterable_expr()}
result = [{d['item']} for {d['item']} in {d['list']}]""",

            "DictComp": lambda: f"""{iterable_expr()}
result = {{str(k): k for k in {d['list']}}}""",

            "SetComp": lambda: f"""{iterable_expr()}
result = {{{d['item']} for {d['item']} in {d['list']}}}""",

            "GeneratorExp": lambda: f"""{iterable_expr()}
result = list({d['item']} for {d['item']} in {d['list']})""",

            "Try": lambda: self._try_tpl(bobj, d, m),
            "ExceptHandler": lambda: self._try_tpl(bobj, d, m),

            "Raise": lambda: (
                f"if not isinstance({mock()}, {bobj}):\n    raise {bobj}('Invalid: ' + str({mock()}))"
                if self._is_exc(bobj) else
                f"{d['value']} = {mock()}\nif {d['value']} is None:\n    raise ValueError(str({bobj}))"
            ),

            "FunctionDef": lambda: (
                f"def {d['func']}({d['item']}: {bobj}) -> {bobj}:\n    result = {bobj}({d['item']})\n    return result"
                if self._is_call(bobj) else
                f"def {d['func']}({d['item']}):\n    \"\"\"Process using {bobj}.\"\"\"\n    return str({d['item']})"
            ),

            "AsyncFunctionDef": lambda: f"""async def {d['func']}({d['item']}):
    result = {bobj}({d['item']}) if callable({bobj}) else str({d['item']})
    return result""",

            "ClassDef": lambda: f"""class {d['class']}:
    data_type = {bobj}
    def __init__(self):
        self.{d['value']} = {mock()}
    def {d['method']}(self):
        return self.{d['value']}""",

            "Return": lambda: f"""def {d['func']}():
    {d['value']} = {mock()}
    return {bobj}({d['value']}) if callable({bobj}) else {d['value']}""",

            "Assign": lambda: f"{d['value']} = {mock()}",
            "AugAssign": lambda: f"{d['value']} = {mock('int')}\n{d['value']} += 1",
            "AnnAssign": lambda: (
                f"{d['value']}: {bobj} = {mock()}" if self._is_call(bobj)
                else f"{d['value']}: str = str({mock()})"
            ),
            "Import": lambda: "import ast",
            "ImportFrom": lambda: "from collections import OrderedDict",

            "Break": lambda: f"""{iterable_expr()}
for {d['item']} in {d['list']}:
    break""",

            "Continue": lambda: f"""{iterable_expr()}
for {d['item']} in {d['list']}:
    if {d['item']}:
        continue
    {d['func']}({d['item']})""",

            "Pass": lambda: f"class {d['class']}:\n    pass",
            "Delete": lambda: f"{d['value']} = {mock()}\ndel {d['value']}",

            "With": lambda: f"""class {d['class']}:
    def __enter__(self): return self
    def __exit__(self, *a): pass
with {d['class']}() as ctx:
    {d['value']} = {mock()}""",

            "AsyncWith": lambda: f"""async def _run():
    class {d['class']}:
        async def __aenter__(self): return self
        async def __aexit__(self, *a): pass
    async with {d['class']}() as ctx:
        {d['value']} = {mock()}""",

            "Call": lambda: (
                f"result = {bobj}({mock()})" if self._is_call(bobj)
                else f"result = str({mock()})"
            ),
            "Attribute": lambda: f"{d['value']} = {mock()}\nresult = {d['value']}.__class__.__name__",
            "Subscript": lambda: f"{iterable_expr()}\nresult = {d['list']}[0]",
            "Lambda": lambda: f"transform = lambda x: {bobj}(x) if callable({bobj}) else x\nresult = transform({mock()})",
            "BinOp": lambda: f"{d['value']} = {mock('int')}\nresult = {d['value']} + 1",
            "BoolOp": lambda: f"{d['value']} = {mock()}\nresult = {d['value']} and True",
            "UnaryOp": lambda: f"{d['value']} = {mock('int')}\nresult = -{d['value']}",
            "Compare": lambda: f"{d['value']} = {mock()}\nresult = {d['value']} == {mock()}",
            "Assert": lambda: f"{d['value']} = {mock()}\nassert {d['value']} is not None",

            "Global": lambda: f"""{d['value']} = {mock()}
def {d['func']}():
    global {d['value']}
    {d['value']} = {mock()}""",

            "Nonlocal": lambda: f"""def {d['func']}_outer():
    {d['value']} = {mock()}
    def inner():
        nonlocal {d['value']}
        {d['value']} = {mock()}
    inner()""",

            "Yield": lambda: f"""def {d['func']}():
    {iterable_expr()}
    for {d['item']} in {d['list']}:
        yield {d['item']}""",

            "YieldFrom": lambda: f"""def {d['func']}():
    {iterable_expr()}
    yield from {d['list']}""",

            "Starred": lambda: f"{iterable_expr()}\nfirst, *rest = {d['list']}",
            "Slice": lambda: f"{iterable_expr()}\nresult = {d['list']}[1:3]",
            "Dict": lambda: f"result = {mock('dict')}",
            "Set": lambda: f"result = {mock('set')}",
        }

        builder = T.get(node)
        if builder is None:
            return None
        try:
            return builder()
        except Exception:
            return None

    def _isolated_tpl(self, node, d):
        """Templates for nodes that take no builtin argument."""
        T = {
            "Pass":     lambda: f"class {d['class']}:\n    pass",
            "Break":    lambda: f"for {d['item']} in range(5):\n    break",
            "Continue": lambda: f"for {d['item']} in range(5):\n    if {d['item']}:\n        continue",
            "Assert":   lambda: f"{d['value']} = 1\nassert {d['value']} is not None",
            "Import":   lambda: "import ast",
            "ImportFrom": lambda: "from collections import OrderedDict",
        }
        builder = T.get(node)
        return builder() if builder else None

    def _try_tpl(self, bobj, d, m):
        if self._is_exc(bobj):
            return (f"try:\n    {d['value']} = {m.get('int', '42')}\n"
                    f"    result = int({d['value']})\n"
                    f"except {bobj} as e:\n    {d['func']}(str(e))")
        return (f"try:\n    result = {bobj}({m.get('list', '[1,2,3]')})\n"
                f"except Exception as e:\n    {d['func']}(str(e))")

    def _apply_wrapper(self, body, wrapper, d):
        if wrapper == "global":
            return body
        elif wrapper == "function":
            return f"def {d['func']}_main():\n{textwrap.indent(body, '    ')}"
        elif wrapper == "method":
            return (f"class {d['class']}Main:\n"
                    f"    def {d['method']}_run(self):\n"
                    f"{textwrap.indent(body, '        ')}")
        return body

    def _verify(self, tree, target):
        tc = getattr(ast, target, None)
        return tc is not None and any(isinstance(n, tc) for n in ast.walk(tree))

    @staticmethod
    def _is_exc(name):
        obj = getattr(builtins_module, name, None)
        return obj is not None and isinstance(obj, type) and issubclass(obj, BaseException)

    @staticmethod
    def _is_call(name):
        obj = getattr(builtins_module, name, None)
        return obj is not None and callable(obj)


# --- Smoke test (pre-filter, no model needed) ---
print("=" * 60)
print("GENERATOR SMOKE TEST (pre-filter examples)")
print("=" * 60)
_smoke_gen = ASTPromptGenerator(DOMAINS)
for _pair in [("For", "list"), ("Try", "ValueError"), ("Pass", "_isolated_")]:
    _samples = _smoke_gen.generate_batch(*_pair, n=3)
    print(f"\n--- {_pair}: {len(_samples)} prompts generated ---")
    if _samples:
        print(_samples[0]["prompt_text"])
print("=" * 60)

## Perplexity Filter (Stage C)

**Atlas 2 §2:** *"Runs a single forward pass for each prompt to calculate the Cross-Entropy Loss. Sorts by lowest loss, truncates to top 100."*

In [ ]:
# version 1.11
# Cell 9 — Perplexity Filter / Stage C (loads model — may take a few minutes)
import torch.nn.functional as F

class PerplexityFilter:

    def __init__(self, model_name: str, device: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"Loading {model_name} on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
        self.model.to(self.device)
        self.model.eval()
        logger.info("Model loaded.")

    def score_prompt(self, prompt_text: str) -> tuple:
        with torch.no_grad():
            input_ids = self.tokenizer(
                prompt_text, return_tensors="pt", truncation=True,
                max_length=512, add_special_tokens=False
            )["input_ids"].to(self.device)
            token_length = input_ids.shape[1]
            if token_length < 2:
                return 50.0, token_length
            logits = self.model(input_ids).logits
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = input_ids[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            ).item()
        return loss, token_length

    def filter_batch(self, prompts, top_k=100, catastrophic_threshold=3.0):
        if not prompts:
            return []
        for p in prompts:
            loss, tok_len = self.score_prompt(p["prompt_text"])
            p["sequence_loss"] = loss
            p["token_length"] = tok_len
        avg_loss = np.mean([p["sequence_loss"] for p in prompts])
        if avg_loss > catastrophic_threshold:
            logger.warning(
                f"Avg loss {avg_loss:.2f} > {catastrophic_threshold}. Discarding cell."
            )
            return []
        # Keep ALL prompts — no sorting by loss
        # Sorting biases toward function wrappers and kills structural variance
        return prompts[:top_k]


pfilter = PerplexityFilter(MODEL_NAME)

## Pipeline (Stage D)

**Atlas 2 §9:** *"Output should be an Apache Parquet file. Schema: `ast_node`, `builtin_obj`, `variation_id`, `prompt_text`, `sequence_loss`, `token_length`, `ast_verified`."*

In [ ]:
# Cell 10 — Pipeline function / Stage D
def run_pipeline(concept_pairs, generator, pfilter, n_generate, n_keep,
                 checkpoint_dir, checkpoint_every=50, run_name=None,
                 catastrophic_threshold=10.0):

    if run_name is None:
        run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")

    all_rows = []
    stats = {
        "run_name": run_name, "mode": MODE,
        "n_ast": len(set(a for a, _ in concept_pairs)),
        "n_builtins": len(set(b for _, b in concept_pairs)),
        "total_pairs": len(concept_pairs),
        "n_generate": n_generate, "n_keep": n_keep,
        "successful_pairs": 0, "failed_pairs": 0,
        "empty_gen_pairs": [], "catastrophic_pairs": [],
        "total_prompts": 0,
    }

    for idx, (ast_node, builtin_obj) in enumerate(tqdm(concept_pairs, desc="Pairs")):
        raw = generator.generate_batch(ast_node, builtin_obj, n=n_generate)
        if not raw:
            stats["failed_pairs"] += 1
            stats["empty_gen_pairs"].append([ast_node, builtin_obj])
            continue

        filtered = pfilter.filter_batch(
            raw, top_k=n_keep, catastrophic_threshold=catastrophic_threshold
        )
        if not filtered:
            stats["failed_pairs"] += 1
            stats["catastrophic_pairs"].append([ast_node, builtin_obj])
            continue

        for var_id, p in enumerate(filtered):
            all_rows.append({
                "ast_node": ast_node, "builtin_obj": builtin_obj,
                "variation_id": var_id, "prompt_text": p["prompt_text"],
                "sequence_loss": p["sequence_loss"],
                "token_length": p["token_length"],
                "ast_verified": p.get("ast_verified", False),
            })
        stats["successful_pairs"] += 1
        stats["total_prompts"] += len(filtered)

        if (idx + 1) % checkpoint_every == 0:
            ckpt = pd.DataFrame(all_rows)
            path = os.path.join(checkpoint_dir, f"{run_name}_ckpt_{idx+1}.parquet")
            ckpt.to_parquet(path, index=False)
            logger.info(f"Checkpoint: {path} ({len(ckpt)} rows)")

    df = pd.DataFrame(all_rows)
    final_path = os.path.join(checkpoint_dir, f"{run_name}_validated_prompts.parquet")
    df.to_parquet(final_path, index=False)

    stats_path = os.path.join(checkpoint_dir, f"{run_name}_stats.json")
    with open(stats_path, "w") as f:
        json.dump(stats, f, indent=2)

    print(f"\n{'='*60}")
    print(f"DONE: {run_name}")
    print(f"  Pairs: {stats['successful_pairs']}/{stats['total_pairs']} succeeded")
    print(f"  Prompts: {stats['total_prompts']}")
    print(f"  Output: {final_path}")
    print(f"{'='*60}")
    return df

## Execute Pipeline

**Run order — required before Cell 11:**
1. Cell 1 — pip install (once per session), then **Runtime → Restart session**
2. Cells 2–8 — configuration, imports, schema, concept matrix, generator
3. Cell 9 — loads the model (takes 1–3 min); **must finish before Cell 11**
4. Cell 10 — defines `run_pipeline()`
5. **Cell 11** — starts the run

> Do not skip Cell 9. If the model is not loaded, Cell 11 will fail immediately.

In [ ]:
# Cell 11 — Execute pipeline
generator = ASTPromptGenerator(DOMAINS)
run_name = {
    "test": "test_5x5x10",
    "small": "small_40x50x50",
    "full": datetime.now().strftime("full_%Y%m%d_%H%M%S"),
}[MODE]

df = run_pipeline(
    concept_pairs=concept_pairs,
    generator=generator, pfilter=pfilter,
    n_generate=N_GENERATE, n_keep=N_KEEP,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_every=CHECKPOINT_EVERY,
    run_name=run_name,
    catastrophic_threshold=CATASTROPHIC_THRESHOLD,
)
df.head(10)

## Quality Inspection

**Acceptance criteria:**
1. ≥80% of pairs produce valid output
2. `ast_verified` ≥95%
3. Loss distribution is left-skewed
4. Genuine lexical variance across variations

In [ ]:
# Cell 12 — Quality report + sample prompts (all modes)
print("=" * 60)
print("QUALITY REPORT")
print("=" * 60)

if df.empty:
    print("\n⚠  DataFrame is empty — no prompts survived the pipeline.")
    print(f"   Pairs attempted : {len(concept_pairs)}")
    print(f"   Check the pipeline stats file in {CHECKPOINT_DIR} for details.")
    # Print stats if available
    import json, os
    stats_path = os.path.join(CHECKPOINT_DIR, f"{run_name}_stats.json")
    if os.path.exists(stats_path):
        with open(stats_path) as f:
            stats = json.load(f)
        print(f"\n   successful_pairs   : {stats.get('successful_pairs')}")
        print(f"   failed_pairs       : {stats.get('failed_pairs')}")
        print(f"   empty_gen_pairs    : {stats.get('empty_gen_pairs')}")
        print(f"   catastrophic_pairs : {stats.get('catastrophic_pairs')}")
else:
    pair_counts = df.groupby(["ast_node", "builtin_obj"]).size().reset_index(name="count")
    total_attempted = len(concept_pairs)
    print(f"\n1. PAIR COVERAGE: {len(pair_counts)} / {total_attempted} "
          f"({100*len(pair_counts)/total_attempted:.0f}%)")

    min_expected = int(N_KEEP * 0.8)
    good = pair_counts[pair_counts["count"] >= min_expected]
    print(f"   Pairs with ≥{min_expected} prompts: {len(good)}")

    verified_rate = df["ast_verified"].mean() * 100
    print(f"\n2. AST VERIFIED: {verified_rate:.1f}%  "
          f"{'✓' if verified_rate >= 95 else '✗ NEEDS ATTENTION'}")

    print(f"\n3. LOSS DISTRIBUTION")
    for stat, val in [("Mean", df["sequence_loss"].mean()),
                      ("Median", df["sequence_loss"].median()),
                      ("Std", df["sequence_loss"].std()),
                      ("Min", df["sequence_loss"].min()),
                      ("Max", df["sequence_loss"].max())]:
        print(f"   {stat}: {val:.3f}")

    print(f"\n4. TOKEN LENGTH: mean={df['token_length'].mean():.0f}, "
          f"range=[{df['token_length'].min()}, {df['token_length'].max()}]")

    print(f"\n5. SAMPLE PROMPTS (post-filter, scored)")
    print("-" * 60)
    n_samples = min(5, len(pair_counts))
    for _, row in pair_counts.sample(n_samples, random_state=0).iterrows():
        sub = df[(df["ast_node"] == row["ast_node"]) & (df["builtin_obj"] == row["builtin_obj"])]
        s = sub.iloc[0]
        print(f"\n  pair : ({s['ast_node']}, {s['builtin_obj']})")
        print(f"  loss : {s['sequence_loss']:.4f}  |  tokens: {s['token_length']}  |  verified: {s['ast_verified']}")
        print(f"  prompt:\n{s['prompt_text']}")
        print("-" * 60)

In [ ]:
# Cell 12 — Loss histogram
import matplotlib.pyplot as plt

if df.empty:
    print("No data to plot — df is empty.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(df["sequence_loss"], bins=30, edgecolor="black", alpha=0.7)
    axes[0].set_xlabel("Sequence Loss")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Loss Distribution (All Prompts)")
    axes[0].axvline(df["sequence_loss"].median(), color="red", linestyle="--",
                    label=f"Median: {df['sequence_loss'].median():.2f}")
    axes[0].legend()

    pair_loss = df.groupby(["ast_node", "builtin_obj"])["sequence_loss"].mean()
    axes[1].hist(pair_loss, bins=20, edgecolor="black", alpha=0.7, color="orange")
    axes[1].set_xlabel("Mean Loss per Pair")
    axes[1].set_title("Per-Pair Mean Loss")

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 13 — Variance check: 5 variations of one pair side-by-side (all modes)
print("VARIANCE CHECK")
print("=" * 60)

if df.empty:
    print("No data — df is empty.")
else:
    sa, sb = df["ast_node"].iloc[0], df["builtin_obj"].iloc[0]
    sub = df[(df["ast_node"] == sa) & (df["builtin_obj"] == sb)]
    print(f"Pair: ({sa}, {sb}) — {len(sub)} variations\n")

    for i, (_, r) in enumerate(sub.head(5).iterrows()):
        print(f"--- Var {r['variation_id']}  loss={r['sequence_loss']:.4f}  tokens={r['token_length']} ---")
        print(r["prompt_text"])
        print()

    firsts = [p.split("\n")[0] for p in sub["prompt_text"]]
    print(f"Unique first lines: {len(set(firsts))}/{len(firsts)}")

## HuggingFace Upload (Full Run Only)

In [ ]:
# Cell 14 — HuggingFace upload (full run only)
if MODE == "full":
    from huggingface_hub import HfApi, login
    login()
    api = HfApi()
    final_path = os.path.join(CHECKPOINT_DIR, f"{run_name}_validated_prompts.parquet")
    api.upload_file(
        path_or_fileobj=final_path,
        path_in_repo="module1/validated_prompts.parquet",
        repo_id=HF_REPO, repo_type="model",
    )
    stats_path = os.path.join(CHECKPOINT_DIR, f"{run_name}_stats.json")
    api.upload_file(
        path_or_fileobj=stats_path,
        path_in_repo="module1/generation_stats.json",
        repo_id=HF_REPO, repo_type="model",
    )
    print(f"Uploaded to https://huggingface.co/{HF_REPO}")
else:
    print(f"Skipping HF upload ({MODE} mode).")

In [ ]:
# Cell 15 — Cleanup intermediate checkpoint files (optional)
import glob
ckpts = glob.glob(os.path.join(CHECKPOINT_DIR, f"{run_name}_ckpt_*.parquet"))
print(f"Found {len(ckpts)} checkpoint files.")
# Uncomment to delete:
# for f in ckpts: os.remove(f)
# print("Deleted.")